In [ ]:
recent = om[om["Year"] >= 2016]["Production_kbd"]
print("Mean 2016-2025:", round(recent.mean()))
print("Min / Max:", round(recent.min()), "/", round(recent.max()))
print("Std dev:", round(recent.std()))

In [ ]:
naive_pred = [train["Production_kbd"].iloc[-1]] * len(test)
mean_pred = [train[train["Year"] >= 2016]["Production_kbd"].mean()] * len(test)

print("Linear trend MAE:", round(mae))
print("Naive (last value) MAE:", round(mean_absolute_error(test["Production_kbd"], naive_pred)))
print("Mean 2016-2022 MAE:", round(mean_absolute_error(test["Production_kbd"], mean_pred)))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np

om = long[(long["Country"] == "Oman") & (long["Year"] >= 2010)].copy()
om["Year"] = om["Year"].astype(int)

# Train on 2010-2022, test on 2023-2025
train = om[om["Year"] <= 2022]
test = om[om["Year"] > 2022]

model = LinearRegression()
model.fit(train[["Year"]], train["Production_kbd"])

pred = model.predict(test[["Year"]])
mae = mean_absolute_error(test["Production_kbd"], pred)
print(f"Test MAE: {mae:.0f} thousand barrels/day")

# Refit on all data, forecast 2026-2030
model.fit(om[["Year"]], om["Production_kbd"])
future = pd.DataFrame({"Year": range(2026, 2031)})
future["Forecast_kbd"] = model.predict(future[["Year"]]).round(0)
print(future)

plt.figure(figsize=(8, 4))
plt.plot(om["Year"], om["Production_kbd"], label="Actual")
plt.plot(future["Year"], future["Forecast_kbd"], "--", label="Forecast")
plt.title("Oman oil production: trend forecast (kbd)")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

q2 = """
SELECT Country, ROUND(Production_kbd, 0) AS kbd
FROM oil_production
WHERE Year = 2025
  AND Country IN ('Oman','Saudi Arabia','United Arab Emirates','Kuwait','Qatar','Iraq','Iran')
ORDER BY kbd DESC
"""
gulf = pd.read_sql(q2, conn)
print(gulf)

oman = long[(long["Country"] == "Oman") & (long["Year"] >= 1990)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(oman["Year"], oman["Production_kbd"])
ax[0].set_title("Oman oil production (thousand barrels/day)")
ax[1].barh(gulf["Country"], gulf["kbd"])
ax[1].set_title("Gulf producers, 2025")
plt.tight_layout()
plt.show()

In [ ]:
print(long[long["Country"].str.contains("Emirates|UAE", case=False)]["Country"].unique())

In [ ]:
import sqlite3

conn = sqlite3.connect("energy.db")
long.to_sql("oil_production", conn, if_exists="replace", index=False)

# Query 1: Oman production, last 10 years
q1 = """
SELECT Year, ROUND(Production_kbd, 0) AS kbd
FROM oil_production
WHERE Country = 'Oman' AND Year >= 2016
ORDER BY Year
"""
print(pd.read_sql(q1, conn))

# Query 2: Oman vs Gulf neighbours, 2025
q2 = """
SELECT Country, ROUND(Production_kbd, 0) AS kbd
FROM oil_production
WHERE Year = 2025
  AND Country IN ('Oman', 'Saudi Arabia', 'UAE', 'Kuwait', 'Qatar', 'Iraq', 'Iran')
ORDER BY kbd DESC
"""
print(pd.read_sql(q2, conn))

In [ ]:
df = pd.read_excel("EI-Stats-Review-ALL-data.xlsx",
                   sheet_name="Oil Production - barrels", header=2)

df = df.iloc[:, :62]                      # keep Country + 1965-2025, drop growth/share columns
df.columns = ["Country"] + list(range(1965, 2026))
df = df.dropna(subset=["Country"])        # remove blank rows

# Convert to long format (one row = country + year)
long = df.melt(id_vars="Country", var_name="Year", value_name="Production_kbd")
long = long.dropna(subset=["Production_kbd"])

print(long.shape)
long[long["Country"] == "Oman"].tail()

In [ ]:
raw = pd.read_excel("EI-Stats-Review-ALL-data.xlsx",
                    sheet_name="Oil Production - barrels", header=None)
raw.head(10)

In [ ]:
import pandas as pd
xl = pd.ExcelFile("EI-Stats-Review-ALL-data.xlsx")
print(xl.sheet_names)